*Section 0: Connect to Google Drive*

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Section 1: Setup and Load Dataset

In this section, I import the required Python libraries, upload the Bank Marketing dataset into Google Colab, and load it into a pandas DataFrame. This step confirms that the dataset is accessible and ready for further inspection and preprocessing.

In [2]:
# Import libraries

import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

print("Libraries imported successfully.")

Libraries imported successfully.


## **1.1** Upload the dataset file

Here, I upload the dataset file into the Colab session. I will upload the main CSV file from the Bank Marketing dataset so that it can be loaded and analysed in the following steps.

In [3]:
# Load dataset from Google Drive

import pandas as pd

file_path = "/content/drive/MyDrive/bank-full.csv"

df = pd.read_csv(file_path, sep=";")

print("Dataset loaded successfully.")
print("File path:", file_path)
print("Shape:", df.shape)
display(df.head())

Dataset loaded successfully.
File path: /content/drive/MyDrive/bank-full.csv
Shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## **1.2** Confirm target column

This cell verifies that the target variable is present and checks the class distribution for the binary classification task.

In [4]:
# Confirming target column

print("Columns:")
print(df.columns.tolist())

print("\nTarget value counts:")
print(df["y"].value_counts())

print("\nTarget proportions:")
print(df["y"].value_counts(normalize=True))

Columns:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

Target value counts:
y
no     39922
yes     5289
Name: count, dtype: int64

Target proportions:
y
no     0.883015
yes    0.116985
Name: proportion, dtype: float64


# Section 2: Initial Data Inspection

In this section, I perform an initial inspection of the Bank Marketing dataset. The aim is to understand the dataset size, feature types, numerical summaries, and categorical variables before carrying out any cleaning, preprocessing, or exploratory data analysis.

In [5]:
# Check overall structure

print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nDetailed info:")
df.info()

Dataset shape: (45211, 17)

Data types:
age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object

Detailed info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month   

## **2.1** Numerical feature summary

This cell provides descriptive statistics for the numerical features in the dataset. It helps identify the range, spread, and central tendency of the continuous and integer-based variables.

In [6]:
# Summary statistics for numerical columns

print("Numerical summary statistics:")
display(df.describe())

Numerical summary statistics:


,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


## **2.2** Separate numerical and categorical features

This cell identifies which variables are numerical and which are categorical. This is important because both groups will require different preprocessing techniques later.

In [7]:
# Identify numerical and categorical columns

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:")
print(numerical_cols)

print("\nCategorical columns:")
print(categorical_cols)

print("\nNumber of numerical columns:", len(numerical_cols))
print("Number of categorical columns:", len(categorical_cols))

Numerical columns:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

Categorical columns:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'y']

Number of numerical columns: 7
Number of categorical columns: 10


## **2.3** Inspect unique values in categorical features

This cell checks the unique values present in each categorical variable. This helps identify important categories, possible inconsistencies, and special values such as 'unknown' that need attention.

In [8]:
# Unique values in categorical columns

for col in categorical_cols:
    print(f"\nColumn: {col}")
    print("Number of unique values:", df[col].nunique())
    print("Unique values:", sorted(df[col].unique()))


Column: job
Number of unique values: 12
Unique values: ['admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown']

Column: marital
Number of unique values: 3
Unique values: ['divorced', 'married', 'single']

Column: education
Number of unique values: 4
Unique values: ['primary', 'secondary', 'tertiary', 'unknown']

Column: default
Number of unique values: 2
Unique values: ['no', 'yes']

Column: housing
Number of unique values: 2
Unique values: ['no', 'yes']

Column: loan
Number of unique values: 2
Unique values: ['no', 'yes']

Column: contact
Number of unique values: 3
Unique values: ['cellular', 'telephone', 'unknown']

Column: month
Number of unique values: 12
Unique values: ['apr', 'aug', 'dec', 'feb', 'jan', 'jul', 'jun', 'mar', 'may', 'nov', 'oct', 'sep']

Column: poutcome
Number of unique values: 4
Unique values: ['failure', 'other', 'success', 'unknown']

Column: y
Number of uniq

## **2.4** Check class balance

This cell reconfirms the class distribution of the target variable. This is useful because class imbalance will influence model training and evaluation choices later in the project.

In [9]:
# Target distribution summary

target_counts = df["y"].value_counts()
target_props = df["y"].value_counts(normalize=True) * 100

print("Target counts:")
print(target_counts)

print("\nTarget percentages:")
print(target_props.round(2))

Target counts:
y
no     39922
yes     5289
Name: count, dtype: int64

Target percentages:
y
no     88.3
yes    11.7
Name: proportion, dtype: float64
